# RECON 1FPS Jupyter Visualizer

This notebook visualizes processed RECON trajectories exported as `jpg + traj_data.pkl`.

## Usage

1. Set `DATA_FOLDERS` to one or more folders such as `datasets/recon_1fps_train`.
2. Run all cells.
3. Use the sliders or play button to move across trajectories and frames.

This view shows the RGB frame and the corresponding 2D trajectory path with current heading.


In [1]:
import json
from html import escape
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import clear_output, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError("Install ipywidgets in this notebook environment to use the interactive controls.") from exc


def _find_recon_datavis_src() -> Path:
    marker = Path("recon_datavis") / "jupyter_visualizer.py"
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        cand = d / "src" / marker
        if cand.is_file():
            return d / "src"
        cand = d / "datasets" / "recon_datavis" / "src" / marker
        if cand.is_file():
            return d / "datasets" / "recon_datavis" / "src"
    raise FileNotFoundError(
        "Could not locate recon_datavis/src (expected recon_datavis/jupyter_visualizer.py). "
        "Open the notebook from the nwm repo or from datasets/recon_datavis."
    )


src_path = _find_recon_datavis_src()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

p = src_path.resolve()
repo_root = p.parent.parent.parent if p.name == "src" and p.parent.name == "recon_datavis" else Path.cwd().resolve()

from recon_datavis.jupyter_visualizer import (
    JupyterProcessedTrajectoryVisualizer,
    collect_processed_trajectory_dirs,
)


In [2]:
DATA_FOLDERS = [
    repo_root / "datasets" / "recon_1fps_train",
    # repo_root / "datasets" / "recon_1fps_test",
]
CAPTION_JSONL = repo_root / "datasets" / "derived" / "phase1_qwen_clean" / "recon_train_1fps_clean.jsonl"
# CAPTION_JSONL = repo_root / "datasets" / "derived" / "phase1_qwen_clean" / "recon_test_1fps_clean.jsonl"

trajectory_dirs = collect_processed_trajectory_dirs([str(folder) for folder in DATA_FOLDERS])
if not trajectory_dirs:
    raise ValueError(f"No processed trajectories found in: {DATA_FOLDERS}")

visualizer = JupyterProcessedTrajectoryVisualizer(trajectory_dirs)
frame_counts = visualizer.get_frame_count_map()
trajectory_dirs = list(visualizer.trajectory_dirs)

caption_index = {}
if CAPTION_JSONL.exists():
    with CAPTION_JSONL.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            key = (record["trajectory_name"], int(record["frame_time"]))
            caption_index[key] = record
else:
    print(f"Caption file not found: {CAPTION_JSONL}")

print(f"Loaded {len(trajectory_dirs)} processed trajectories")
print(trajectory_dirs[0])
print(f"Frames in first trajectory: {frame_counts[0]}")
print(f"Loaded {len(caption_index)} caption records")


Loaded 9468 processed trajectories
/workspace/nwm/datasets/recon_1fps_train/jackal_2019-08-02-16-23-30_0_r00
Frames in first trajectory: 7
Loaded 53395 caption records


In [3]:
trajectory_slider = widgets.IntSlider(value=0, min=0, max=len(trajectory_dirs) - 1, step=1, description="Trajectory")
frame_slider = widgets.IntSlider(value=0, min=0, max=frame_counts[0] - 1, step=1, description="Frame")
play = widgets.Play(value=0, min=0, max=frame_counts[0] - 1, interval=300, description="Play")
widgets.jslink((play, "value"), (frame_slider, "value"))
output = widgets.Output()
caption_box = widgets.HTML()

def _sync_frame_range(*_):
    max_frame = frame_counts[trajectory_slider.value] - 1
    frame_slider.max = max_frame
    play.max = max_frame
    if frame_slider.value > max_frame:
        frame_slider.value = max_frame

def _render(*_):
    with output:
        clear_output(wait=True)
        fig, _ = visualizer.render(file_idx=trajectory_slider.value, timestep=frame_slider.value)
        display(fig)
        plt.close(fig)

    trajectory_name = Path(trajectory_dirs[trajectory_slider.value]).name
    frame_time = frame_slider.value
    record = caption_index.get((trajectory_name, frame_time))
    if record is None:
        caption_box.value = (
            f"<div style='padding:10px 12px;border:1px solid #ddd;border-radius:8px;margin-top:8px;'>"
            f"<b>Trajectory</b>: <code>{escape(trajectory_name)}</code><br>"
            f"<b>Frame</b>: <code>{frame_time}</code><br>"
            f"<b>Caption</b>: not found"
            f"</div>"
        )
    else:
        clean_text = escape(record.get("clean_text") or record.get("raw_caption") or "")
        raw_caption = escape(record.get("raw_caption") or "")
        extra_raw = "" if raw_caption == clean_text or not raw_caption else f"<br><b>Raw Caption</b>: {raw_caption}"
        caption_box.value = (
            f"<div style='padding:10px 12px;border:1px solid #ddd;border-radius:8px;margin-top:8px;line-height:1.5;'>"
            f"<b>Trajectory</b>: <code>{escape(trajectory_name)}</code><br>"
            f"<b>Frame</b>: <code>{frame_time}</code><br>"
            f"<b>Clean Caption</b>: {clean_text}"
            f"{extra_raw}"
            f"</div>"
        )

trajectory_slider.observe(_sync_frame_range, names="value")
trajectory_slider.observe(_render, names="value")
frame_slider.observe(_render, names="value")

_sync_frame_range()
_render()

display(widgets.VBox([
    widgets.HBox([trajectory_slider]),
    widgets.HBox([play, frame_slider]),
    output,
    caption_box,
]))
